In [1]:
import requests
import pandas as pd
import numpy as np
from difflib import get_close_matches

HEADERS = {'User-Agent': 'Mozilla/5.0'}

# ── 1. FPL Data ──────────────────────────────────────────────────────────────
fpl_data = requests.get("https://fantasy.premierleague.com/api/bootstrap-static/", headers=HEADERS).json()
fpl_players = pd.DataFrame(fpl_data['elements'])
fpl_teams   = pd.DataFrame(fpl_data['teams'])
fixtures    = pd.DataFrame(requests.get("https://fantasy.premierleague.com/api/fixtures/", headers=HEADERS).json())

# ── 2. Understat Data ────────────────────────────────────────────────────────
def get_understat_players(season="2024", league="EPL"):
    url = "https://understat.com/main/getPlayersStats/"
    response = requests.post(url, data={'league': league, 'season': season}, headers=HEADERS)
    df = pd.DataFrame(response.json()['players'])
    for col in ['goals','xG','assists','xA','shots','key_passes','games','time','npg','npxG']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

understat = get_understat_players()

# ── 3. Clean FPL ─────────────────────────────────────────────────────────────
team_map = fpl_teams.set_index('id')['name'].to_dict()

fpl = fpl_players[[
    'id','web_name','first_name','second_name','team','element_type',
    'now_cost','total_points','minutes','starts','form',
    'goals_scored','assists','clean_sheets','goals_conceded',
    'expected_goals','expected_assists','expected_goal_involvements',
    'expected_goals_per_90','expected_assists_per_90',
    'expected_goals_conceded_per_90','clean_sheets_per_90',
    'points_per_game','selected_by_percent','bonus','bps',
    'influence','creativity','threat','ict_index',
    'chance_of_playing_next_round','status','news'
]].copy()

fpl['team_name'] = fpl['team'].map(team_map)
fpl['position']  = fpl['element_type'].map({1:'GKP',2:'DEF',3:'MID',4:'FWD'})
fpl['price']     = fpl['now_cost'] / 10
fpl['full_name'] = fpl['first_name'] + ' ' + fpl['second_name']

for col in ['expected_goals','expected_assists','expected_goal_involvements',
            'expected_goals_per_90','expected_assists_per_90','form',
            'expected_goals_conceded_per_90','clean_sheets_per_90',
            'influence','creativity','threat','ict_index',
            'points_per_game','selected_by_percent']:
    fpl[col] = pd.to_numeric(fpl[col], errors='coerce')

fpl['90s_played'] = fpl['minutes'] / 90

# ── 4. Fixture Difficulty ────────────────────────────────────────────────────
future = fixtures[fixtures['finished'] == False].copy()
team_strength = fpl_teams.set_index('id')[['strength_overall_home','strength_overall_away']]

def get_fdr(team_id, n=5):
    home = future[future['team_h'] == team_id].head(n).copy()
    away = future[future['team_a'] == team_id].head(n).copy()
    home['opp_str'] = home['team_a'].map(team_strength['strength_overall_away'])
    away['opp_str'] = away['team_h'].map(team_strength['strength_overall_home'])
    all_fix = pd.concat([home[['event','opp_str']], away[['event','opp_str']]]).sort_values('event').head(n)
    return all_fix['opp_str'].mean() if len(all_fix) > 0 else np.nan

fdr_map = {tid: get_fdr(tid) for tid in fpl_teams['id']}
fpl['fdr_5gw'] = fpl['team'].map(fdr_map)
fdr_min, fdr_max = fpl['fdr_5gw'].min(), fpl['fdr_5gw'].max()
fpl['fdr_norm'] = (fpl['fdr_5gw'] - fdr_min) / (fdr_max - fdr_min)

# ── 5. Merge Understat ───────────────────────────────────────────────────────
def match_players(fpl_names, understat_names):
    us_lower = {n.lower(): n for n in understat_names}
    matches = {}
    for name in fpl_names:
        result = get_close_matches(name.lower(), us_lower.keys(), n=1, cutoff=0.6)
        if result:
            matches[name] = us_lower[result[0]]
    return matches

name_map = match_players(fpl['full_name'].tolist(), understat['player_name'].tolist())
fpl['understat_match'] = fpl['full_name'].map(name_map)

us_idx = understat.set_index('player_name')
fpl['us_xG']      = fpl['understat_match'].map(us_idx['xG'])
fpl['us_xA']      = fpl['understat_match'].map(us_idx['xA'])
fpl['us_npxG']    = fpl['understat_match'].map(us_idx['npxG'])
fpl['us_shots']   = fpl['understat_match'].map(us_idx['shots'])
fpl['us_xGChain'] = fpl['understat_match'].map(us_idx['xGChain'])

print(f"Understat matched: {fpl['understat_match'].notna().sum()}/{len(fpl)} players")

# ── 6. xPts Model ────────────────────────────────────────────────────────────
def calc_xpts(row):
    pos = row['position']
    nineties = row['90s_played']
    if nineties < 1:
        return np.nan

    xg  = row['us_npxG'] if pd.notna(row['us_npxG']) else row['expected_goals']
    xa  = row['us_xA']   if pd.notna(row['us_xA'])   else row['expected_assists']
    xg_per90 = (xg or 0) / nineties
    xa_per90 = (xa or 0) / nineties
    xgc_per90 = row['expected_goals_conceded_per_90'] or 0

    goal_pts = {'FWD':4,'MID':5,'DEF':6,'GKP':6}.get(pos, 4)
    cs_prob  = max(0, 1 - xgc_per90)
    cs_pts   = {'GKP':6,'DEF':4,'MID':1,'FWD':0}.get(pos, 0) * cs_prob
    bonus_per90 = (row['bonus'] or 0) / nineties

    xpts = (xg_per90 * goal_pts) + (xa_per90 * 3) + cs_pts + 2 + bonus_per90
    fixture_factor = 1 + 0.15 * (1 - (row['fdr_norm'] or 0.5))
    return round(xpts * fixture_factor, 2)

fpl['xpts_per90'] = fpl.apply(calc_xpts, axis=1)
fpl['value']      = fpl['xpts_per90'] / fpl['price']

# ── 7. Results ───────────────────────────────────────────────────────────────
model = fpl[fpl['xpts_per90'].notna()].sort_values('xpts_per90', ascending=False)

print("\n=== TOP 20 BY xPts/90 ===")
print(model.head(20)[['web_name','position','team_name','price','xpts_per90','value','us_xG','us_xA']].to_string())

print("\n=== BEST VALUE (≥£5m) ===")
print(model[model['price']>=5].sort_values('value',ascending=False).head(15)[
    ['web_name','position','team_name','price','xpts_per90','value']].to_string())

print("\n=== BY POSITION ===")
for pos in ['GKP','DEF','MID','FWD']:
    print(f"\n-- {pos} --")
    print(model[model['position']==pos].head(5)[
        ['web_name','team_name','price','xpts_per90','value']].to_string())

Understat matched: 565/819 players

=== TOP 20 BY xPts/90 ===
       web_name position       team_name  price  xpts_per90      value      us_xG      us_xA
272      M.Sarr      DEF         Chelsea    4.5      104.88  23.306667  12.308559  10.056549
376        Röhl      MID         Everton    5.0       33.05   6.610000  23.954593   3.581227
27      Havertz      FWD         Arsenal    7.3       32.83   4.497260  11.830621   2.948872
483    Trafford      GKP        Man City    4.5       28.62   6.360000  12.219387   3.737535
564       Wissa      FWD       Newcastle    7.3       22.70   3.109589  20.731060   3.449474
476        Isak      FWD       Liverpool   10.3       21.35   2.072816  22.356988   5.448704
433       Piroe      FWD           Leeds    4.9       18.06   3.685714   8.863605   5.618519
736       Adama      MID        West Ham    5.1       17.95   3.519608   5.658593   6.424516
628        Wood      FWD   Nott'm Forest    7.2       13.56   1.883333  15.638655   3.044111
69      

In [13]:
import time

# ── Fetch historical GW data for all players ──────────────────────────────────
def get_player_history(player_id):
    url = f"https://fantasy.premierleague.com/api/element-summary/{player_id}/"
    r = requests.get(url, headers=HEADERS)
    if r.status_code != 200:
        return pd.DataFrame()
    data = r.json()
    df = pd.DataFrame(data['history'])
    df['player_id'] = player_id
    return df

# Only fetch available players with enough minutes
eligible = fpl[fpl['minutes'] > 90]['id'].tolist()
print(f"Fetching history for {len(eligible)} players...")

all_history = []
for i, pid in enumerate(eligible):
    df = get_player_history(pid)
    if not df.empty:
        all_history.append(df)
    if i % 50 == 0:
        print(f"  {i}/{len(eligible)} done...")
        time.sleep(0.5)  # be polite to the API

history_df = pd.concat(all_history, ignore_index=True)

# Add player metadata
meta = fpl_players[['id','web_name','element_type','team','now_cost']].copy()
meta.columns = ['player_id','web_name','element_type','team','now_cost']
pos_map = {1:'GKP',2:'DEF',3:'MID',4:'FWD'}
meta['position'] = meta['element_type'].map(pos_map)
team_name_map = fpl_teams.set_index('id')['name'].to_dict()
meta['team_name'] = meta['team'].map(team_name_map)
history_df = history_df.merge(meta, on='player_id', how='left')

# Clean types
for col in ['total_points','minutes','expected_goals','expected_assists',
            'expected_goal_involvements','expected_goals_conceded',
            'goals_scored','assists','clean_sheets','bonus','bps']:
    if col in history_df.columns:
        history_df[col] = pd.to_numeric(history_df[col], errors='coerce')

print(f"\nHistory shape: {history_df.shape}")
print(f"GWs covered: {sorted(history_df['round'].unique())}")
print(history_df.head())

Fetching history for 452 players...
  0/452 done...
  50/452 done...
  100/452 done...
  150/452 done...
  200/452 done...
  250/452 done...
  300/452 done...
  350/452 done...
  400/452 done...
  450/452 done...

History shape: (12333, 48)
GWs covered: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28)]
   element  fixture  opponent_team  total_points  was_home  \
0        1        9             14            10     False   
1        1       11             11             6      True   
2        1       25             12             2     False   
3        1       31             16             6      True   
4        1       41             13             2    

In [20]:
def build_features(history_df, fpl_teams):
    df = history_df.copy().sort_values(['player_id', 'round'])

    # ── Rolling form features (lagged to avoid leakage) ──────────────────────
    for window in [3, 5, 10]:
        df[f'pts_avg_{window}'] = df.groupby('player_id')['total_points'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )
        df[f'xg_avg_{window}'] = df.groupby('player_id')['expected_goals'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )
        df[f'xa_avg_{window}'] = df.groupby('player_id')['expected_assists'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )
        df[f'xgi_avg_{window}'] = df.groupby('player_id')['expected_goal_involvements'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )
        df[f'mins_avg_{window}'] = df.groupby('player_id')['minutes'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )
        df[f'bonus_avg_{window}'] = df.groupby('player_id')['bonus'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )

    # ── Rotation risk (minutes consistency) ──────────────────────────────────
    df['minutes_ratio'] = (df['mins_avg_5'] / 90).clip(0, 1)

    # ── Home/away ─────────────────────────────────────────────────────────────
    df['is_home'] = df['was_home'].astype(int)

    # ── Fixture difficulty ────────────────────────────────────────────────────
    # Build from opponent team strength
    team_strength = fpl_teams.set_index('id')[['strength_overall_home', 'strength_overall_away',
                                                'strength_attack_home', 'strength_attack_away',
                                                'strength_defence_home', 'strength_defence_away']].copy()

    def opp_attack(row):
        t = row['opponent_team']
        if t not in team_strength.index:
            return np.nan
        return team_strength.loc[t, 'strength_attack_home'] if not row['was_home'] else team_strength.loc[t, 'strength_attack_away']

    def opp_defence(row):
        t = row['opponent_team']
        if t not in team_strength.index:
            return np.nan
        return team_strength.loc[t, 'strength_defence_home'] if row['was_home'] else team_strength.loc[t, 'strength_defence_away']

    df['opp_attack_strength']  = df.apply(opp_attack, axis=1)
    df['opp_defence_strength'] = df.apply(opp_defence, axis=1)

    # Normalise 0-1
    df['opp_attack_norm']  = (df['opp_attack_strength']  - df['opp_attack_strength'].min())  / (df['opp_attack_strength'].max()  - df['opp_attack_strength'].min())
    df['opp_defence_norm'] = (df['opp_defence_strength'] - df['opp_defence_strength'].min()) / (df['opp_defence_strength'].max() - df['opp_defence_strength'].min())

    # ── Opponent rolling goals conceded (last 5 GWs) ─────────────────────────
    team_gc = df.groupby(['opponent_team', 'round'])['team_h_score'].mean().reset_index()
    team_gc.columns = ['team_id', 'round', 'goals_scored_avg']
    team_gc['opp_gc_rolling5'] = team_gc.groupby('team_id')['goals_scored_avg'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
    team_gc_renamed = team_gc[['team_id', 'round', 'opp_gc_rolling5']].rename(columns={'team_id': '_merge_team_id'})
    df = df.merge(team_gc_renamed,
              left_on=['opponent_team', 'round'], right_on=['_merge_team_id', 'round'], how='left').drop(columns='_merge_team_id')

    # ── Position encoding ─────────────────────────────────────────────────────
    pos_order = {'GKP': 0, 'DEF': 1, 'MID': 2, 'FWD': 3}
    df['position_ord'] = df['position'].map(pos_order)
    pos_dummies = pd.get_dummies(df['position'], prefix='pos')
    df = pd.concat([df, pos_dummies], axis=1)

    # ── Price tier ────────────────────────────────────────────────────────────
    df['price_norm'] = (df['value'] / 10 - 3.5) / (15 - 3.5)  # normalise to 0-1

    # ── Clean sheet proxy ─────────────────────────────────────────────────────
    df['cs_rolling5'] = df.groupby('player_id')['clean_sheets'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )

    # ── Target variable ───────────────────────────────────────────────────────
    df['target'] = df['total_points']

    return df

features_df = build_features(history_df, fpl_teams)

feature_cols = [
    'pts_avg_3','pts_avg_5','pts_avg_10',
    'xg_avg_3','xg_avg_5','xg_avg_10',
    'xa_avg_3','xa_avg_5','xa_avg_10',
    'xgi_avg_3','xgi_avg_5','xgi_avg_10',
    'mins_avg_3','mins_avg_5','mins_avg_10',
    'bonus_avg_3','bonus_avg_5','bonus_avg_10',
    'minutes_ratio','is_home',
    'opp_attack_norm','opp_defence_norm','opp_gc_rolling5',
    'position_ord','price_norm','cs_rolling5',
    'pos_GKP','pos_DEF','pos_MID','pos_FWD',
]

print(f"Feature matrix: {features_df.shape}")
print(f"\nFeature columns ({len(feature_cols)}):")
print(feature_cols)
print(f"\nMissing values:\n{features_df[feature_cols].isna().sum()[features_df[feature_cols].isna().sum() > 0]}")
print(f"\nSample:\n{features_df[['web_name','round','position'] + feature_cols[:8] + ['target']].head(10).to_string()}")

Feature matrix: (12333, 81)

Feature columns (30):
['pts_avg_3', 'pts_avg_5', 'pts_avg_10', 'xg_avg_3', 'xg_avg_5', 'xg_avg_10', 'xa_avg_3', 'xa_avg_5', 'xa_avg_10', 'xgi_avg_3', 'xgi_avg_5', 'xgi_avg_10', 'mins_avg_3', 'mins_avg_5', 'mins_avg_10', 'bonus_avg_3', 'bonus_avg_5', 'bonus_avg_10', 'minutes_ratio', 'is_home', 'opp_attack_norm', 'opp_defence_norm', 'opp_gc_rolling5', 'position_ord', 'price_norm', 'cs_rolling5', 'pos_GKP', 'pos_DEF', 'pos_MID', 'pos_FWD']

Missing values:
pts_avg_3          452
pts_avg_5          452
pts_avg_10         452
xg_avg_3           452
xg_avg_5           452
xg_avg_10          452
xa_avg_3           452
xa_avg_5           452
xa_avg_10          452
xgi_avg_3          452
xgi_avg_5          452
xgi_avg_10         452
mins_avg_3         452
mins_avg_5         452
mins_avg_10        452
bonus_avg_3        452
bonus_avg_5        452
bonus_avg_10       452
minutes_ratio      452
opp_gc_rolling5    403
cs_rolling5        452
dtype: int64

Sample:
  web_na

In [21]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

FEATURE_COLS = [
    'pts_avg_3','pts_avg_5','pts_avg_10',
    'xg_avg_3','xg_avg_5','xg_avg_10',
    'xa_avg_3','xa_avg_5','xa_avg_10',
    'xgi_avg_3','xgi_avg_5',
    'mins_avg_3','mins_avg_5',
    'bonus_avg_3','bonus_avg_5',
    'minutes_ratio','is_home',
    'opp_attack_norm','opp_defence_norm',
    'price_norm','cs_rolling5',
]

# Position-specific features (added on top of base)
POS_EXTRA = {
    'GKP': ['cs_rolling5'],
    'DEF': ['cs_rolling5', 'opp_attack_norm'],
    'MID': ['xgi_avg_3', 'xgi_avg_5', 'xa_avg_3'],
    'FWD': ['xg_avg_3', 'xg_avg_5', 'xgi_avg_3'],
}

# ── 1. Train separate model per position ─────────────────────────────────────
def train_position_models(df):
    models = {}
    results = {}

    for pos in ['GKP', 'DEF', 'MID', 'FWD']:
        pos_df = df[df['position'] == pos].copy()
        pos_df = pos_df.dropna(subset=FEATURE_COLS + ['target']).sort_values('round')
        pos_df['opp_gc_rolling5'] = pos_df.get('opp_gc_rolling5', pd.Series(dtype=float)).fillna(
            pos_df.get('opp_gc_rolling5', pd.Series(dtype=float)).mean()
        ) if 'opp_gc_rolling5' in pos_df.columns else 0

        split_gw = int(pos_df['round'].quantile(0.8))
        train = pos_df[pos_df['round'] < split_gw]
        test  = pos_df[pos_df['round'] >= split_gw]

        X_train, y_train = train[FEATURE_COLS], train['target']
        X_test,  y_test  = test[FEATURE_COLS],  test['target']

        model = XGBRegressor(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.04,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            random_state=42,
            verbosity=0,
        )
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

        preds        = model.predict(X_test)
        mae          = mean_absolute_error(y_test, preds)
        baseline_mae = mean_absolute_error(y_test, [y_train.mean()] * len(y_test))

        models[pos]  = model
        results[pos] = {'mae': mae, 'baseline': baseline_mae, 'test': test, 'preds': preds}

        print(f"{pos}: MAE={mae:.3f}  Baseline={baseline_mae:.3f}  "
              f"Improvement={((baseline_mae-mae)/baseline_mae*100):.1f}%  "
              f"(n_train={len(train)}, n_test={len(test)})")

    return models, results

models, results = train_position_models(features_df)

# ── 2. Volatility score ───────────────────────────────────────────────────────
features_df['volatility_5'] = features_df.groupby('player_id')['total_points'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=3).std()
)
features_df['volatility_10'] = features_df.groupby('player_id')['total_points'].transform(
    lambda x: x.shift(1).rolling(10, min_periods=5).std()
)

# ── 3. Generate next-GW predictions for all current players ──────────────────
def predict_next_gw(features_df, models, fpl, fpl_teams, fixtures):
    future     = fixtures[fixtures['finished'] == False].copy()
    next_gw    = int(future['event'].min())
    team_strength = fpl_teams.set_index('id')

    # Get each player's latest feature row (most recent GW)
    latest = (features_df.sort_values('round')
                         .groupby('player_id')
                         .last()
                         .reset_index())

    # Attach next GW fixture
    def get_next_fixture(team_id):
        home = future[future['team_h'] == team_id].sort_values('event').head(1)
        away = future[future['team_a'] == team_id].sort_values('event').head(1)
        if home.empty and away.empty:
            return None
        if home.empty:
            row = away.iloc[0]
            return {'is_home': False, 'opponent': row['team_h'], 'difficulty': row['team_a_difficulty'], 'gw': row['event']}
        if away.empty:
            row = home.iloc[0]
            return {'is_home': True,  'opponent': row['team_a'], 'difficulty': row['team_h_difficulty'], 'gw': row['event']}
        # pick whichever is next
        if home.iloc[0]['event'] <= away.iloc[0]['event']:
            row = home.iloc[0]
            return {'is_home': True,  'opponent': row['team_a'], 'difficulty': row['team_h_difficulty'], 'gw': row['event']}
        else:
            row = away.iloc[0]
            return {'is_home': False, 'opponent': row['team_h'], 'difficulty': row['team_a_difficulty'], 'gw': row['event']}

    rows = []
    for _, player in latest.iterrows():
        pid  = player['player_id']
        pos  = player['position']
        if pos not in models:
            continue

        fpl_row = fpl[fpl['id'] == pid]
        if fpl_row.empty:
            continue
        fpl_row = fpl_row.iloc[0]

        team_id = fpl_row['team']
        fix     = get_next_fixture(team_id)
        if fix is None:
            continue

        # Update fixture-dependent features for next GW
        feat = player[FEATURE_COLS].copy()
        feat['is_home'] = int(fix['is_home'])

        opp_id = fix['opponent']
        if opp_id in team_strength.index:
            opp = team_strength.loc[opp_id]
            raw_att  = opp['strength_attack_home']  if not fix['is_home'] else opp['strength_attack_away']
            raw_def  = opp['strength_defence_home'] if fix['is_home']     else opp['strength_defence_away']
            all_att  = features_df['opp_attack_strength'].dropna()
            all_def  = features_df['opp_defence_strength'].dropna()
            feat['opp_attack_norm']  = (raw_att - all_att.min()) / (all_att.max() - all_att.min())
            feat['opp_defence_norm'] = (raw_def - all_def.min()) / (all_def.max() - all_def.min())

        feat_df   = pd.DataFrame([feat])[FEATURE_COLS].fillna(0)
        xp        = float(models[pos].predict(feat_df)[0])
        xp        = max(0, round(xp, 2))

        # Volatility (last known)
        vol = player.get('volatility_5', np.nan)
        vol = 0 if pd.isna(vol) else round(float(vol), 2)

        # Captain score: xP + 0.3 * volatility (rewards high-ceiling players)
        captain_score = round(xp + 0.3 * vol, 2)

        opp_name = team_strength.loc[opp_id, 'short_name'] if opp_id in team_strength.index else '?'

        rows.append({
            'player_id':     pid,
            'player':        fpl_row['web_name'],
            'position':      pos,
            'team':          fpl_row['team_name'],
            'price':         fpl_row['price'],
            'status':        fpl_row['status'],
            'gw':            fix['gw'],
            'opponent':      opp_name,
            'is_home':       fix['is_home'],
            'difficulty':    fix['difficulty'],
            'xP':            xp,
            'volatility':    vol,
            'captain_score': captain_score,
            'ownership':     fpl_row['selected_by_percent'],
        })

    pred_df = pd.DataFrame(rows).sort_values('xP', ascending=False)
    return pred_df, next_gw

predictions, next_gw = predict_next_gw(features_df, models, fpl, fpl_teams, fixtures)

# ── Output ────────────────────────────────────────────────────────────────────
print(f"\n=== GW{next_gw} xP PREDICTIONS ===\n")
print(predictions[predictions['status'] == 'a']
      .head(30)[['player','position','team','price','opponent','is_home',
                 'difficulty','xP','volatility','captain_score','ownership']]
      .to_string(index=False))

print(f"\n=== CAPTAIN PICKS (GW{next_gw}) ===")
print(predictions[predictions['status'] == 'a']
      .sort_values('captain_score', ascending=False)
      .head(10)[['player','position','team','price','xP','volatility','captain_score']]
      .to_string(index=False))

print(f"\n=== BY POSITION ===")
for pos in ['GKP','DEF','MID','FWD']:
    print(f"\n-- {pos} --")
    print(predictions[(predictions['position'] == pos) & (predictions['status'] == 'a')]
          .head(8)[['player','team','price','opponent','xP','captain_score']]
          .to_string(index=False))

GKP: MAE=1.746  Baseline=1.983  Improvement=12.0%  (n_train=642, n_test=189)
DEF: MAE=1.792  Baseline=2.026  Improvement=11.5%  (n_train=3409, n_test=1012)
MID: MAE=1.659  Baseline=1.955  Improvement=15.1%  (n_train=4063, n_test=1210)
FWD: MAE=1.917  Baseline=1.976  Improvement=3.0%  (n_train=1079, n_test=277)

=== GW28 xP PREDICTIONS ===

     player position        team  price opponent  is_home  difficulty   xP  volatility  captain_score  ownership
   Martinez      GKP Aston Villa    5.1      WOL    False           2 7.36        3.35           8.37        4.7
     Senesi      DEF Bournemouth    4.9      SUN     True           2 6.33        2.55           7.09       16.2
   Dúbravka      GKP     Burnley    4.0      BRE     True           3 6.10        1.73           6.62       32.4
   Truffert      DEF Bournemouth    4.5      SUN     True           2 6.06        2.17           6.71        2.1
     Konaté      DEF   Liverpool    5.5      WHU     True           2 5.95        4.83       

In [22]:
!pip install pulp --quiet

In [23]:
import pulp
import pandas as pd
import numpy as np

def optimize_squad(
    predictions_df,
    budget=100.0,
    existing_squad_ids=None,
    free_transfers=1,
    n_gws=1,
):
    """
    predictions_df must have:
        player_id, player, position, team, price, xP, status
    For multi-GW pass xP as sum of xP across N gameweeks
    """
    df = predictions_df.copy()

    # Only available players
    df = df[df['status'] == 'a'].dropna(subset=['xP']).reset_index(drop=True)

    # Map position to int for constraints
    pos_map = {'GKP': 1, 'DEF': 2, 'MID': 3, 'FWD': 4}
    df['pos_int'] = df['position'].map(pos_map)

    # Index by player_id for fast lookup
    df = df.set_index('player_id')
    pids = df.index.tolist()

    prob = pulp.LpProblem("FPL_Optimizer", pulp.LpMaximize)

    # ── Decision variables ────────────────────────────────────────────────────
    selected  = {p: pulp.LpVariable(f"sel_{p}",   cat="Binary") for p in pids}
    starting  = {p: pulp.LpVariable(f"start_{p}", cat="Binary") for p in pids}
    captain   = {p: pulp.LpVariable(f"cap_{p}",   cat="Binary") for p in pids}
    vice_cap  = {p: pulp.LpVariable(f"vc_{p}",    cat="Binary") for p in pids}

    # Transfer penalty vars
    if existing_squad_ids:
        transfer_in = {p: pulp.LpVariable(f"tin_{p}", cat="Binary") for p in pids}

    # ── Objective: maximise xP + captain bonus - transfer penalty ─────────────
    base_xp = pulp.lpSum(starting[p] * df.loc[p, 'xP'] for p in pids)
    cap_bonus = pulp.lpSum(captain[p] * df.loc[p, 'xP'] for p in pids)  # captain doubles

    if existing_squad_ids:
        # 4pt hit per extra transfer beyond free transfers
        extra_transfers = pulp.lpSum(
            transfer_in[p] for p in pids if p not in existing_squad_ids
        ) - free_transfers
        transfer_penalty = 4 * extra_transfers
        prob += base_xp + cap_bonus - transfer_penalty
    else:
        prob += base_xp + cap_bonus

    # ── Squad constraints ─────────────────────────────────────────────────────
    prob += pulp.lpSum(selected.values()) == 15
    prob += pulp.lpSum(starting.values()) == 11
    prob += pulp.lpSum(captain.values())  == 1
    prob += pulp.lpSum(vice_cap.values()) == 1

    for p in pids:
        prob += captain[p]  <= starting[p]
        prob += vice_cap[p] <= starting[p]
        prob += starting[p] <= selected[p]
        prob += captain[p] + vice_cap[p] <= 1  # can't be both

    # ── Position constraints (squad: 2 GKP, 5 DEF, 5 MID, 3 FWD) ────────────
    for pos_int, count in [(1, 2), (2, 5), (3, 5), (4, 3)]:
        pos_pids = df[df['pos_int'] == pos_int].index.tolist()
        prob += pulp.lpSum(selected[p] for p in pos_pids) == count

    # ── Starting 11 formation constraints ─────────────────────────────────────
    gkp_pids = df[df['pos_int'] == 1].index.tolist()
    def_pids  = df[df['pos_int'] == 2].index.tolist()
    mid_pids  = df[df['pos_int'] == 3].index.tolist()
    fwd_pids  = df[df['pos_int'] == 4].index.tolist()

    prob += pulp.lpSum(starting[p] for p in gkp_pids) == 1  # exactly 1 GKP starts
    prob += pulp.lpSum(starting[p] for p in def_pids) >= 3  # min 3 DEF
    prob += pulp.lpSum(starting[p] for p in mid_pids) >= 2  # min 2 MID
    prob += pulp.lpSum(starting[p] for p in fwd_pids) >= 1  # min 1 FWD

    # ── Budget ────────────────────────────────────────────────────────────────
    prob += pulp.lpSum(selected[p] * df.loc[p, 'price'] for p in pids) <= budget

    # ── Max 3 per team ────────────────────────────────────────────────────────
    for team in df['team'].unique():
        team_pids = df[df['team'] == team].index.tolist()
        prob += pulp.lpSum(selected[p] for p in team_pids) <= 3

    for p in pids:
        prob += captain[p]  <= starting[p]
        prob += vice_cap[p] <= starting[p]
        prob += starting[p] <= selected[p]
        prob += captain[p] + vice_cap[p] <= 1  # can't be both

    # ADD HERE ↓
    gkp_pids_cap = df[df['position'] == 'GKP'].index.tolist()
    for p in gkp_pids_cap:
        prob += captain[p] == 0
        prob += vice_cap[p] == 0

    # ── Transfer constraints ──────────────────────────────────────────────────
    if existing_squad_ids:
        for p in pids:
            if p not in existing_squad_ids:
                prob += transfer_in[p] >= selected[p]
            else:
                prob += transfer_in[p] == 0  # already owned, not a transfer in

    # ── Solve ─────────────────────────────────────────────────────────────────
    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    if prob.status != 1:
        print("No optimal solution found:", pulp.LpStatus[prob.status])
        return None

    squad_ids    = [p for p in pids if selected[p].value() == 1]
    starter_ids  = [p for p in pids if starting[p].value()  == 1]
    captain_id   = next(p for p in pids if captain[p].value()  == 1)
    vice_cap_id  = next(p for p in pids if vice_cap[p].value() == 1)

    # ── Build result dataframe ────────────────────────────────────────────────
    squad_df = df.loc[squad_ids].copy().reset_index()
    squad_df['is_starting'] = squad_df['player_id'].isin(starter_ids)
    squad_df['is_captain']  = squad_df['player_id'] == captain_id
    squad_df['is_vice']     = squad_df['player_id'] == vice_cap_id
    squad_df['role'] = squad_df.apply(
        lambda r: '★ C' if r['is_captain'] else ('VC' if r['is_vice'] else ('START' if r['is_starting'] else 'BENCH')), axis=1
    )
    if existing_squad_ids:
        squad_df['is_transfer'] = ~squad_df['player_id'].isin(existing_squad_ids)
        n_transfers = squad_df['is_transfer'].sum()
        hits = max(0, n_transfers - free_transfers) * 4
    else:
        squad_df['is_transfer'] = False
        n_transfers, hits = 0, 0

    squad_df = squad_df.sort_values(['is_starting', 'pos_int'], ascending=[False, True])
    total_xp   = round(pulp.value(prob.objective), 2)
    total_cost = round(squad_df['price'].sum(), 1)

    return {
        'squad':        squad_df,
        'captain':      df.loc[captain_id, 'player'],
        'vice_captain': df.loc[vice_cap_id, 'player'],
        'total_xP':     total_xp,
        'total_cost':   total_cost,
        'n_transfers':  n_transfers,
        'hits':         hits,
    }


def print_squad(result):
    if result is None:
        return
    sq = result['squad']
    print(f"\n{'='*65}")
    print(f" OPTIMAL SQUAD   xP={result['total_xP']}   Cost=£{result['total_cost']}m")
    print(f" Captain: {result['captain']}   VC: {result['vice_captain']}")
    if result['n_transfers'] > 0:
        print(f" Transfers: {result['n_transfers']}  Hit: -{result['hits']}pts")
    print(f"{'='*65}")

    for section, label in [(True, 'STARTING XI'), (False, 'BENCH')]:
        print(f"\n  {label}")
        print(f"  {'Player':<20} {'Pos':<5} {'Team':<18} {'Price':>6} {'xP':>6}  {'Role'}")
        print(f"  {'-'*65}")
        sub = sq[sq['is_starting'] == section].sort_values('pos_int')
        for _, r in sub.iterrows():
            transfer_flag = ' ← IN' if r['is_transfer'] else ''
            print(f"  {r['player']:<20} {r['position']:<5} {r['team']:<18} "
                  f"£{r['price']:>4.1f}  {r['xP']:>5.2f}  {r['role']}{transfer_flag}")


# ── Run: fresh squad (no existing team) ──────────────────────────────────────
result = optimize_squad(predictions, budget=100.0)
print_squad(result)

# ── Run: with existing squad + 1 free transfer ───────────────────────────────
# Replace with your actual player IDs from predictions['player_id']
# existing = [id1, id2, ...]  # your current 15 player IDs
# result_transfer = optimize_squad(predictions, budget=100.0,
#                                  existing_squad_ids=existing,
#                                  free_transfers=1)
# print_squad(result_transfer)


 OPTIMAL SQUAD   xP=69.74   Cost=£90.3m
 Captain: Senesi   VC: B.Fernandes

  STARTING XI
  Player               Pos   Team                Price     xP  Role
  -----------------------------------------------------------------
  Martinez             GKP   Aston Villa        £ 5.1   7.36  START
  Senesi               DEF   Bournemouth        £ 4.9   6.33  ★ C
  Truffert             DEF   Bournemouth        £ 4.5   6.06  START
  Konaté               DEF   Liverpool          £ 5.5   5.95  START
  Groß                 MID   Brighton           £ 5.5   5.53  START
  Adli                 MID   Bournemouth        £ 5.4   5.47  START
  Semenyo              MID   Man City           £ 8.1   5.44  START
  B.Fernandes          MID   Man Utd            £ 9.9   5.41  VC
  Szoboszlai           MID   Liverpool          £ 6.8   5.21  START
  Watkins              FWD   Aston Villa        £ 8.6   5.63  START
  Raúl                 FWD   Fulham             £ 6.1   5.02  START

  BENCH
  Player             

In [26]:
# ── END-TO-END WEEKLY PIPELINE ────────────────────────────────────────────────
# Run this every GW to get fresh predictions and optimal squad

import requests, time
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from difflib import get_close_matches
import pulp

HEADERS = {'User-Agent': 'Mozilla/5.0'}

# ── 1. FETCH ──────────────────────────────────────────────────────────────────
print("Fetching FPL data...")
fpl_data  = requests.get("https://fantasy.premierleague.com/api/bootstrap-static/", headers=HEADERS).json()
fpl_players = pd.DataFrame(fpl_data['elements'])
fpl_teams   = pd.DataFrame(fpl_data['teams'])
fixtures    = pd.DataFrame(requests.get("https://fantasy.premierleague.com/api/fixtures/", headers=HEADERS).json())
understat   = get_understat_players()  # already defined above

# Current and next GW
future   = fixtures[fixtures['finished'] == False].copy()
next_gw  = int(future['event'].min())
print(f"Next GW: {next_gw}")

# ── 2. PLAYER HISTORY ─────────────────────────────────────────────────────────
print("Fetching player histories...")
eligible = fpl_players[fpl_players['minutes'] > 90]['id'].tolist()
all_history = []
for i, pid in enumerate(eligible):
    r = requests.get(f"https://fantasy.premierleague.com/api/element-summary/{pid}/", headers=HEADERS)
    if r.status_code == 200:
        df = pd.DataFrame(r.json()['history'])
        if not df.empty:
            df['player_id'] = pid
            all_history.append(df)
    if i % 50 == 0:
        print(f"  {i}/{len(eligible)}...")
        time.sleep(0.3)

history_df = pd.concat(all_history, ignore_index=True)
meta = fpl_players[['id','web_name','element_type','team','now_cost']].copy()
meta.columns = ['player_id','web_name','element_type','team_id','now_cost']
pos_map = {1:'GKP',2:'DEF',3:'MID',4:'FWD'}
meta['position'] = meta['element_type'].map(pos_map)
team_name_map = fpl_teams.set_index('id')['name'].to_dict()
meta['team_name'] = meta['team_id'].map(team_name_map)
history_df = history_df.merge(meta, on='player_id', how='left')
for col in ['total_points','minutes','expected_goals','expected_assists',
            'expected_goal_involvements','expected_goals_conceded',
            'goals_scored','assists','clean_sheets','bonus']:
    if col in history_df.columns:
        history_df[col] = pd.to_numeric(history_df[col], errors='coerce')
print(f"History: {history_df.shape}")

# ── 3. FEATURES ───────────────────────────────────────────────────────────────
print("Building features...")
features_df = build_features(history_df, fpl_teams)
features_df['volatility_5'] = features_df.groupby('player_id')['total_points'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=3).std()
)
features_df['volatility_10'] = features_df.groupby('player_id')['total_points'].transform(
    lambda x: x.shift(1).rolling(10, min_periods=5).std()
)
# ── 4. TRAIN ──────────────────────────────────────────────────────────────────
print("Training models...")
models, results = train_position_models(features_df)
for pos, r in results.items():
    print(f"  {pos}: MAE={r['mae']:.3f}")

# ── 5. PREDICT ────────────────────────────────────────────────────────────────
print("Generating predictions...")
fpl_clean = fpl_players.copy()
fpl_clean['team_name'] = fpl_clean['team'].map(team_name_map)
fpl_clean['position']  = fpl_clean['element_type'].map(pos_map)
fpl_clean['price']     = fpl_clean['now_cost'] / 10

predictions, next_gw = predict_next_gw(features_df, models, fpl_clean, fpl_teams, fixtures)

# ── 6. OPTIMIZE ───────────────────────────────────────────────────────────────
print("Optimizing squad...")
result = optimize_squad(predictions, budget=100.0)
print_squad(result)

# ── 7. SUMMARY ────────────────────────────────────────────────────────────────
print(f"\n=== TOP CAPTAIN PICKS GW{next_gw} ===")
print(predictions[predictions['status'] == 'a']
      .sort_values('captain_score', ascending=False)
      .head(10)[['player','position','team','price','xP','volatility','captain_score']]
      .to_string(index=False))

print(f"\n=== BEST VALUE DIFFERENTIALS (own% < 15) ===")
print(predictions[(predictions['status'] == 'a') &
                  (pd.to_numeric(predictions['ownership'], errors='coerce') < 15)]
      .sort_values('captain_score', ascending=False)
      .head(10)[['player','position','team','price','xP','ownership','captain_score']]
      .to_string(index=False))

Fetching FPL data...
Next GW: 29
Fetching player histories...
  0/452...
  50/452...
  100/452...
  150/452...
  200/452...
  250/452...
  300/452...
  350/452...
  400/452...
  450/452...
History: (12333, 48)
Building features...
Training models...
GKP: MAE=1.746  Baseline=1.983  Improvement=12.0%  (n_train=642, n_test=189)
DEF: MAE=1.800  Baseline=2.033  Improvement=11.5%  (n_train=3409, n_test=1012)
MID: MAE=1.668  Baseline=1.963  Improvement=15.0%  (n_train=4063, n_test=1210)
FWD: MAE=1.924  Baseline=1.984  Improvement=3.0%  (n_train=1079, n_test=277)
  GKP: MAE=1.746
  DEF: MAE=1.800
  MID: MAE=1.668
  FWD: MAE=1.924
Generating predictions...
Optimizing squad...

 OPTIMAL SQUAD   xP=75.9   Cost=£86.0m
 Captain: Semenyo   VC: Raúl

  STARTING XI
  Player               Pos   Team                Price     xP  Role
  -----------------------------------------------------------------
  Pickford             GKP   Everton            £ 5.6   7.07  START
  Konaté               DEF   Liverpo